 Name:- GAURAV MUKESH SHARMA TYAGI
 BITSID:- 2026AJ05215
 Machine Learning - Assignment 2

 ## Dataset Details
**Dataset:** UCI Spambase  
**Source:** https://archive.ics.uci.edu/dataset/94/spambase

This dataset was chosen because the assignment requires a public classification dataset with **at least 500 rows and 12 features**. Spambase contains **4,601 rows and 57 input features**, so it meets both requirements.

The **target column** is `spam`: **1 means spam**, and **0 means not spam**. The same dataset is used for all five models.

## 1. Imports

Import the Python libraries needed for the assignment.

In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.metrics import matthews_corrcoef

## 2. Loading the Dataset

The downloaded dataset contains `spambase.data` and `spambase.names`.

The data file does not have column names. The feature names were taken from `spambase.names`, and `spam` was added as the target column.

In [2]:
feature_names = []

with open("spambase.names", "r") as file:
    for line in file:
        if ":" in line and line.strip().endswith("continuous."):
            name = line.split(":")[0].strip()
            feature_names.append(name)

column_names = feature_names + ["spam"]
data = pd.read_csv("spambase.data", header=None, names=column_names)

print("Rows and columns:", data.shape)
print("Number of features:", len(feature_names))
print("First five rows:")
display(data.head())

Rows and columns: (4601, 58)
Number of features: 57
First five rows:


,word_freq_make,word_freq_address,word_freq_all,word_freq_3d,word_freq_our,word_freq_over,word_freq_remove,word_freq_internet,word_freq_order,word_freq_mail,...,char_freq_;,char_freq_(,char_freq_[,char_freq_!,char_freq_$,char_freq_#,capital_run_length_average,capital_run_length_longest,capital_run_length_total,spam
0,0.00,0.64,0.64,0.0,0.32,0.00,0.00,0.00,0.00,0.00,...,0.00,0.000,0.0,0.778,0.000,0.000,3.756,61,278,1
1,0.21,0.28,0.50,0.0,0.14,0.28,0.21,0.07,0.00,0.94,...,0.00,0.132,0.0,0.372,0.180,0.048,5.114,101,1028,1
2,0.06,0.00,0.71,0.0,1.23,0.19,0.19,0.12,0.64,0.25,...,0.01,0.143,0.0,0.276,0.184,0.010,9.821,485,2259,1
3,0.00,0.00,0.00,0.0,0.63,0.00,0.31,0.63,0.31,0.63,...,0.00,0.137,0.0,0.137,0.000,0.000,3.537,40,191,1
4,0.00,0.00,0.00,0.0,0.63,0.00,0.31,0.63,0.31,0.63,...,0.00,0.135,0.0,0.135,0.000,0.000,3.537,40,191,1


## 3. Checking the Dataset

Check the number of rows, missing values, duplicate rows, negative values and the largest feature values. Also check the number of spam and non-spam emails.

In [3]:
print("Number of rows:", len(data))
print("Number of features:", len(feature_names))
print("Missing values:", data.isnull().sum().sum())
print("Duplicate rows:", data.duplicated().sum())
print("Negative feature values:", (data[feature_names] < 0).sum().sum())
print("Target values:")
print(data["spam"].value_counts())
print("Largest values in the features:")
print(data[feature_names].max().sort_values(ascending=False).head(10))

Number of rows: 4601
Number of features: 57
Missing values: 0
Duplicate rows: 391
Negative feature values: 0
Target values:
spam
0    2788
1    1813
Name: count, dtype: int64
Largest values in the features:
capital_run_length_total      15841.000
capital_run_length_longest     9989.000
capital_run_length_average     1102.500
word_freq_3d                     42.810
word_freq_george                 33.330
char_freq_!                      32.478
word_freq_edu                    22.050
word_freq_re                     21.420
word_freq_hp                     20.830
word_freq_85                     20.000
dtype: float64


## 4. Cleaning the Dataset

No missing values or negative feature values were found. Exact duplicate rows were removed.

Some features contain much larger values than others. These values were kept because they may help identify spam. Normalisation is applied to the models that need it in Step 6.

In [4]:
if data.isnull().sum().sum() != 0:
    raise ValueError("Please check the missing values before continuing.")

if (data[feature_names] < 0).any().any():
    raise ValueError("Please check negative values before continuing.")

if set(data["spam"].unique()) != {0, 1}:
    raise ValueError("The target column must contain only 0 and 1.")

rows_before = len(data)
data = data.drop_duplicates().reset_index(drop=True)
print("Rows before cleaning:", rows_before)
print("Duplicates removed:", rows_before - len(data))
print("Rows after cleaning:", len(data))

Rows before cleaning: 4601
Duplicates removed: 391
Rows after cleaning: 4210


## 5. Splitting the Dataset

A **70% training and 30% testing** split was selected. The assignment does not specify a split percentage.

The same split is used for all five models.

In [5]:
X = data[feature_names]
y = data["spam"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

Training rows: 2947
Testing rows: 1263


## 6. Normalising the Features

Some features contain much larger values than others. **MinMaxScaler** was used for Logistic Regression, KNN and SVM to put feature values on a similar scale.

`make_pipeline` was used to apply scaling using only the training data. Decision Tree and Naive Bayes use the original values.

## 7. Creating the Five Models

These are the five classification models required by the assignment.

In [6]:
models = {
    "Logistic Regression": make_pipeline(MinMaxScaler(), LogisticRegression(max_iter=2000)),
    "Decision Tree": DecisionTreeClassifier(max_depth=8, random_state=42),
    "KNN": make_pipeline(MinMaxScaler(), KNeighborsClassifier(n_neighbors=5)),
    "Naive Bayes": GaussianNB(),
    "Support Vector Machine": make_pipeline(
    MinMaxScaler(),
    SVC(kernel="rbf", random_state=42)
)
}

for name in models:
    print(name)

Logistic Regression
Decision Tree
KNN
Naive Bayes
Support Vector Machine


## 8. Training the Models

Train all five models using the same training data.

In [7]:
for name, model in models.items():
    model.fit(X_train, y_train)
    print(name, "trained")

Logistic Regression trained
Decision Tree trained
KNN trained
Naive Bayes trained
Support Vector Machine trained


## 9. Calculating the Six Metrics

For each model, calculate **Accuracy, AUC, Precision, Recall, F1 and MCC** using the test data. For AUC, probability scores are used for four models and decision scores are used for SVM.

In [8]:

results = []

for name, model in models.items():
    predicted = model.predict(X_test)

    if name == "Support Vector Machine":
        spam_probability = model.decision_function(X_test)
    else:
        spam_probability = model.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predicted),
        "AUC": roc_auc_score(y_test, spam_probability),
        "Precision": precision_score(y_test, predicted),
        "Recall": recall_score(y_test, predicted),
        "F1": f1_score(y_test, predicted),
        "MCC": matthews_corrcoef(y_test, predicted)
    })

comparison = pd.DataFrame(results)
display(comparison.round(4))

,Model,Accuracy,AUC,Precision,Recall,F1,MCC
0,Logistic Regression,0.8931,0.9529,0.9261,0.7956,0.8559,0.7774
1,Decision Tree,0.9010,0.9247,0.8844,0.8651,0.8746,0.7930
2,KNN,0.8876,0.9477,0.8851,0.8254,0.8542,0.7641
3,Naive Bayes,0.8321,0.9477,0.7219,0.9425,0.8176,0.6875
4,Support Vector Machine,0.9272,0.9662,0.9310,0.8829,0.9063,0.8476


## 10. Model Metrics

Show all six metrics separately for each model.

In [9]:
for _, row in comparison.iterrows():
    print(row["Model"])
    print("Accuracy:", round(row["Accuracy"], 4))
    print("AUC:", round(row["AUC"], 4))
    print("Precision:", round(row["Precision"], 4))
    print("Recall:", round(row["Recall"], 4))
    print("F1:", round(row["F1"], 4))
    print("MCC:", round(row["MCC"], 4))

Logistic Regression
Accuracy: 0.8931
AUC: 0.9529
Precision: 0.9261
Recall: 0.7956
F1: 0.8559
MCC: 0.7774
Decision Tree
Accuracy: 0.901
AUC: 0.9247
Precision: 0.8844
Recall: 0.8651
F1: 0.8746
MCC: 0.793
KNN
Accuracy: 0.8876
AUC: 0.9477
Precision: 0.8851
Recall: 0.8254
F1: 0.8542
MCC: 0.7641
Naive Bayes
Accuracy: 0.8321
AUC: 0.9477
Precision: 0.7219
Recall: 0.9425
F1: 0.8176
MCC: 0.6875
Support Vector Machine
Accuracy: 0.9272
AUC: 0.9662
Precision: 0.931
Recall: 0.8829
F1: 0.9063
MCC: 0.8476


## 11. Saving CSV Files

The test data and model comparison results are saved as CSV files. The test data will be used later in the Streamlit application. It contains all 57 input features and the target column.

In [10]:
comparison.to_csv("model_comparison.csv", index=False)

test_data = X_test.copy()
test_data["spam"] = y_test
test_data.to_csv("test_data.csv", index=False)

print("Saved model_comparison.csv")
print("Saved test_data.csv")
print("Test data rows and columns:", test_data.shape)

Saved model_comparison.csv
Saved test_data.csv
Test data rows and columns: (1263, 58)


## 12. Saving the Trained Models

The five trained models are saved in the `model` folder. These files will be used later in the Streamlit application.

In [11]:
import os
import joblib

os.makedirs("model", exist_ok=True)

for name, model in models.items():
    filename = name.replace(" ", "_") + ".joblib"
    joblib.dump(model, "model/" + filename)
    print(filename, "saved")

Logistic_Regression.joblib saved
Decision_Tree.joblib saved
KNN.joblib saved
Naive_Bayes.joblib saved
Support_Vector_Machine.joblib saved
